<a href="https://colab.research.google.com/github/milijonierius19/The_Lightmapper/blob/master/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install/upgrade packages — Colab usually has these but pin known-good versions
!pip install -q transformers torch sentencepiece

# Quick check
import transformers, torch
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

transformers: 5.0.0
torch: 2.10.0+cpu
CUDA available: False


In [ ]:
from nltk.corpus import reuters
from nltk import trigrams
from collections import defaultdict
import nltk

nltk.download('reuters', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Placeholder for model: nested dict of counts
model = defaultdict(lambda: defaultdict(lambda: 0))

# Count co-occurrence frequencies of all trigrams in Reuters
for sentence in reuters.sents():
    for w1, w2, w3 in trigrams(sentence, pad_right=True, pad_left=True):
        model[(w1, w2)][w3] += 1

# Convert raw counts to probabilities
for w1_w2 in model:
    total_count = float(sum(model[w1_w2].values()))
    for w3 in model[w1_w2]:
        model[w1_w2][w3] /= total_count

print("Model built! Number of unique (w1, w2) contexts:", len(model))

# --- Use 3 different two-word combinations to predict the next word ---

def top_predictions(w1, w2, n=10):
    """Return top-n most likely next words after (w1, w2)."""
    preds = dict(model[w1, w2])
    return sorted(preds.items(), key=lambda x: x[1], reverse=True)[:n]

print("\nTop predictions after 'today the':")
for word, prob in top_predictions("today", "the"):
    print(f"  {word:20s} {prob:.4f}")

print("\nTop predictions after 'the price':")
for word, prob in top_predictions("the", "price"):
    print(f"  {word:20s} {prob:.4f}")

print("\nTop predictions after 'the company':")
for word, prob in top_predictions("the", "company"):
    print(f"  {word:20s} {prob:.4f}")

Model built! Number of unique (w1, w2) contexts: 398630

Top predictions after 'today the':
  company              0.1667
  price                0.1111
  public               0.0556
  European             0.0556
  Bank                 0.0556
  emirate              0.0556
  overseas             0.0556
  newspaper            0.0556
  Turkish              0.0556
  increase             0.0556

Top predictions after 'the price':
  of                   0.3209
  it                   0.0558
  to                   0.0558
  for                  0.0512
  .                    0.0233
  at                   0.0233
  adjustment           0.0233
  is                   0.0186
  ,                    0.0186
  paid                 0.0140

Top predictions after 'the company':
  '                    0.2452
  said                 0.2195
  .                    0.0829
  ,                    0.0407
  to                   0.0355
  has                  0.0227
  would                0.0222
  is                   0

In [ ]:
import random

def generate_sentence(w1, w2, max_words=20):
    text = [w1, w2]
    for _ in range(max_words):
        candidates = list(model[(w1, w2)].keys())
        probs = list(model[(w1, w2)].values())
        if not candidates:
            break
        next_word = random.choices(candidates, weights=probs)[0]
        if next_word is None:
            break
        text.append(next_word)
        w1, w2 = w2, next_word
    return " ".join(text)

random.seed(42)
print(generate_sentence("today", "the"))
print(generate_sentence("the", "price"))
print(generate_sentence("the", "company"))

today the Turkish General Staff , said that RMJ employed some 300 mln dlrs of new conversion assignments at Geonex ' s
the price drop .
the company ' s 1986 / 87 from 550 mln stg on Allied - Lyons won control of Shaw ' s suspension


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load BART summarization model directly (no pipeline needed)
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def summarize(text, max_length=60, min_length=25):
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=min_length,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True,
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

article1 = """
The James Webb Space Telescope has revolutionized our understanding of the early universe.
Since its launch in December 2021, it has captured images of galaxies that formed within the
first few hundred million years after the Big Bang. These observations are challenging existing
theories of galaxy formation because the early galaxies appear far more massive and structured
than expected. Scientists at NASA and the European Space Agency are now analyzing spectra
of distant exoplanet atmospheres, searching for biosignatures such as water vapor, methane,
and carbon dioxide. The telescope's infrared capabilities allow it to peer through cosmic dust
clouds that block visible light, revealing star-forming regions previously hidden from view.
"""

article2 = """
Quantum computing has reached a critical milestone as researchers at Google and IBM announced
new processors capable of running algorithms beyond the reach of classical supercomputers.
Quantum bits, or qubits, leverage superposition and entanglement to process information in
fundamentally different ways than traditional binary systems. However, qubits are extremely
sensitive to environmental noise, requiring temperatures near absolute zero to maintain coherence.
Industry experts predict that practical applications in drug discovery, cryptography, and
optimization problems could become reality within the next decade, although large-scale
fault-tolerant quantum computers still face significant engineering challenges.
"""

article3 = """
The Roman Empire at its peak in the second century AD spanned three continents and ruled over
approximately 70 million people. Emperor Trajan extended the empire's borders to their greatest
extent, conquering Dacia and parts of Mesopotamia. The Romans built an extensive road network
of over 400,000 kilometers, connecting cities from Britain to North Africa. This infrastructure
allowed for rapid military deployment, efficient trade, and cultural exchange. Roman law,
engineering achievements like aqueducts and concrete construction, and the Latin language
left a lasting legacy that still influences modern Western civilization today.
"""

for i, art in enumerate([article1, article2, article3], 1):
    print(f"--- Summary of Article {i} ---")
    print(summarize(art, max_length=60, min_length=25))
    print()

# Try longer summaries — that's the "change the number of words" exercise
print("\n--- Longer summary of Article 1 (max 120 words) ---")
print(summarize(article1, max_length=120, min_length=60))

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


--- Summary of Article 1 ---
 The James Webb Space Telescope has captured images of galaxies that formed within the first few hundred million years after the Big Bang . The telescope's infrared capabilities allow it to peer through cosmic dust that block visible light .

--- Summary of Article 2 ---
 Google and IBM announce new processors capable of running algorithms beyond the reach of classical supercomputers . Experts predict practical applications in drug discovery, cryptography, and optimization problems could become reality within the next decade .

--- Summary of Article 3 ---
 The Roman Empire at its peak in the second century AD spanned three continents and ruled over approximately 70 million people . Emperor Trajan extended the empire's borders to their greatest extent, conquering Dacia and parts of Mesopotamia . The Romans built an extensive road network of over 400,


--- Longer summary of Article 1 (max 120 words) ---
 The James Webb Space Telescope has captured images of

In [ ]:
from transformers import pipeline

# --- FIX: lab calls it "summarizer" (wrong) and uses undefined "sentiment_pipeline"
sentiment_pipeline = pipeline("sentiment-analysis",
                              model="distilbert-base-uncased-finetuned-sst-2-english")

feedback = """
I bought these wireless headphones two weeks ago and the sound quality is absolutely amazing.
The bass is deep, the battery lasts a full day, and the noise cancellation works perfectly on
the train. Best purchase I have made this year!
"""

result = sentiment_pipeline(feedback)
print(f"Feedback: {feedback.strip()}")
print(f"\nSentiment: {result[0]['label']}, Confidence: {result[0]['score']:.2f}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Feedback: I bought these wireless headphones two weeks ago and the sound quality is absolutely amazing.
The bass is deep, the battery lasts a full day, and the noise cancellation works perfectly on
the train. Best purchase I have made this year!

Sentiment: POSITIVE, Confidence: 1.00


In [ ]:
from transformers import pipeline

# --- FIX: lab uses both "summarizer" and "sentiment_analyzer" (undefined). Use one consistent name.
sentiment_analyzer = pipeline("sentiment-analysis",
                              model="distilbert-base-uncased-finetuned-sst-2-english")

sentences = [
    "The phone screen cracked after one week even though I was very careful with it, terrible quality.",
    "This restaurant has the best pasta I've ever tasted; the service was friendly and the prices were fair.",
    "The package arrived on time. The product matches the description shown on the website."
]

for sentence in sentences:
    sentiment = sentiment_analyzer(sentence)[0]
    print(f"Sentence : {sentence}")
    print(f"Sentiment: {sentiment['label']} ({sentiment['score']:.2f})")
    print("-" * 80)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentence : The phone screen cracked after one week even though I was very careful with it, terrible quality.
Sentiment: NEGATIVE (1.00)
--------------------------------------------------------------------------------
Sentence : This restaurant has the best pasta I've ever tasted; the service was friendly and the prices were fair.
Sentiment: POSITIVE (1.00)
--------------------------------------------------------------------------------
Sentence : The package arrived on time. The product matches the description shown on the website.
Sentiment: NEGATIVE (0.75)
--------------------------------------------------------------------------------


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

# Set pad token (GPT-2 has none by default — silences a warning)
tokenizer.pad_token = tokenizer.eos_token

prompts = [
    "Once upon a time in a magical forest,",
    "The astronaut opened the airlock and stepped onto the red surface of Mars,",
    "On the morning of her first day at the new job, Maria discovered a secret door"
]

for i, prompt in enumerate(prompts, 1):
    inputs = tokenizer.encode(prompt, return_tensors='pt')
    outputs = model.generate(
        inputs,
        max_length=100,
        num_return_sequences=1,
        no_repeat_ngram_size=2,   # avoids "the the the" loops
        do_sample=True,           # sampling -> more creative output
        top_k=50,
        top_p=0.95,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    story = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"--- Prompt {i} ---")
    print(story)
    print()

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--- Prompt 1 ---
Once upon a time in a magical forest, some men used to fight together. Their battles were fought for the lives of many warriors and had nothing to do with politics or the affairs of the king. However, many of them knew of each other and loved their battle. These men were known as warriors because they were the true warriors of their realm. They always tried their best. Many would not survive a duel, and most would fight with swords or spears.

Many people in the city of

--- Prompt 2 ---
The astronaut opened the airlock and stepped onto the red surface of Mars, the space station, and into orbit. When the astronauts reached the Earth orbit, they went back into the shuttle capsule to await an end to their journey on Mars.

In the spacecraft, on the surface in the middle of the Martian atmosphere, there were three small lakes that allowed for the passage of water and oxygen, as well as the opportunity to experience the world of space. At a minimum, those lakes would be

-

In [ ]:
from transformers import pipeline, set_seed

generator = pipeline('text-generation', model='gpt2')
set_seed(42)  # reproducible output

prompts = [
    "The future of renewable energy depends on",
    "In the year 2150, humanity finally discovered",
    "The recipe for the perfect chocolate cake includes"
]

for i, prompt in enumerate(prompts, 1):
    generated = generator(prompt, max_length=80, num_return_sequences=1,
                          truncation=True,
                          pad_token_id=50256)  # eos_token_id for gpt2
    print(f"--- Prompt {i} ---")
    print(generated[0]['generated_text'])
    print()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingf

--- Prompt 1 ---
The future of renewable energy depends on how we use our energy supplies in the future, and what is the future of fossil fuels we can invest in for the future.

With the help of its own research, we have shown how to use our own energy, without putting in too much effort, to create clean energy.

So, take the money you earn from your business, and invest it into clean energy. If we use that money wisely, it will give us a lot more to invest in our business and help us grow.

To learn more about how we can benefit from your investment, please visit:

https://energy.gov/

What's new in this year's portfolio?

The new portfolio includes:

The clean energy energy portfolio aims to grow the clean energy sector in the United States by 2040.

The clean energy portfolio is designed to deliver a net financial benefit to the U.S. public by cutting greenhouse gas emissions from our power plants. The clean energy portfolio will provide significant reductions in global greenhouse g

Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Prompt 2 ---
In the year 2150, humanity finally discovered the dark side of our universe, and it's the dark side that makes us so damn lonely. There are certain things we have to do to survive, and one of them is to take responsibility for our actions. This is one of the last ways humanity has managed to avoid the inevitable disaster that would become the End. Humanity is not a perfect system, but it is a very good one.

For those of you who have not heard, the End of Humanity was a major event in the past that was a massive catastrophe for humanity, and it's still one of the most important events in the history of humanity, yet it's still being considered the epic event that has brought the entire planet back to a state of darkness. It is a huge event that was, and still is, a major event in the history of humanity.

In the year 2150, humanity finally discovered the dark side of our universe, and it's the dark side that makes us so damn lonely. There are certain things we have to 